# 12A — Point-in-Time S&P 500 Universe and Frozen Design

This notebook constructs the four historical S&P 500 universes and identifier bridge for the critical-minerals experiment. It does not download filings, call OpenAI, or open subsequent returns. A decision gate must pass before corpus acquisition begins.

## 1. Environment, paths, and immutable design

In [ ]:
from pathlib import Path
from getpass import getpass
import hashlib, json, os
import numpy as np
import pandas as pd
from IPython.display import display
HERE=Path.cwd().resolve()
if HERE.name=='sp500_critical_minerals': EXT=HERE; GEO=HERE.parent; ROOT=HERE.parents[1]
elif HERE.name=='geopolitical_exposure': GEO=HERE; ROOT=HERE.parent; EXT=GEO/'sp500_critical_minerals'
elif HERE.name=='Asset_Embeddings': ROOT=HERE; GEO=ROOT/'geopolitical_exposure'; EXT=GEO/'sp500_critical_minerals'
else: ROOT=(HERE/'Projects'/'Asset_Embeddings').resolve(); GEO=ROOT/'geopolitical_exposure'; EXT=GEO/'sp500_critical_minerals'
CONFIG=EXT/'config'; DATA=EXT/'data'; PROCESSED=DATA/'processed'; RESULTS=EXT/'results'; MANIFESTS=EXT/'manifests'
for p in [CONFIG,PROCESSED,RESULTS,MANIFESTS]: p.mkdir(parents=True,exist_ok=True)
def sha256(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
design_path=CONFIG/'sp500_experiment_design_v1.json'
design=json.loads(design_path.read_text()); design_hash=sha256(design_path)
vintages=pd.DataFrame(design['vintages']); vintages['cutoff']=pd.to_datetime(vintages.cutoff)
display(pd.DataFrame([{'design_version':design['design_version'],'sha256':design_hash,'vintages':len(vintages),'outcomes_accessed':False,'api_calls_made':False}]))
display(vintages[['label','cutoff','investable_start']])

## 2. Connect to WRDS

Credentials are used only for the current kernel and are not stored.

In [ ]:
import wrds
connection_ok=False
if 'db' in globals() and getattr(db,'connection',None) is not None:
    try: db.raw_sql('select 1 as connection_test'); connection_ok=True
    except Exception:
        try: db.close()
        except Exception: pass
if not connection_ok:
    username=input('WRDS username: ').strip(); password=getpass('WRDS password: ')
    db=wrds.Connection(wrds_username=username,wrds_password=password); del password
    db.raw_sql('select 1 as connection_test')
print('WRDS connection is active.')

## 3. Discover the historical S&P 500 constituent table

The code searches accessible CRSP schemas rather than assuming a particular legacy table name.

In [ ]:
catalog=db.raw_sql("""
select table_schema,table_name,column_name,ordinal_position
from information_schema.columns
where table_schema in ('crsp','crsp_a_indexes','crsp_a_stock')
  and (position('sp500' in lower(table_name)) > 0 or position('500list' in lower(table_name)) > 0)
order by table_schema,table_name,ordinal_position
""")
catalog.to_parquet(MANIFESTS/'sp500_candidate_table_catalog.parquet',index=False)
table_summary=(catalog.groupby(['table_schema','table_name'],as_index=False).agg(columns=('column_name','count'),column_names=('column_name',lambda x:', '.join(x))))
display(table_summary)
if table_summary.empty: raise RuntimeError('No S&P 500 constituent table was found in the accessible CRSP schemas.')

## 4. Probe candidate tables and freeze the selected source

A valid source must contain a security identifier and either dated observations or membership start/end dates.

In [ ]:
probes=[]; valid=[]
for row in table_summary.itertuples(index=False):
    qualified=f'{row.table_schema}.{row.table_name}'
    cols=catalog[(catalog.table_schema==row.table_schema)&(catalog.table_name==row.table_name)].column_name.str.lower().tolist()
    permno_col=next((c for c in ['permno','lpermno'] if c in cols),None)
    observation_col=next((c for c in ['caldt','date','mthcaldt'] if c in cols),None)
    start_col=next((c for c in ['start','startdt','begdt','fromdate'] if c in cols),None)
    end_col=next((c for c in ['ending','enddt','thrudate'] if c in cols),None)
    try:
        sample=db.raw_sql(f'select * from {qualified} limit 5')
        queryable=True; error=None
        if permno_col and (observation_col or (start_col and end_col)):
            spec={'table':qualified,'permno_col':permno_col,'observation_col':observation_col,'start_col':start_col,'end_col':end_col}; valid.append(spec)
    except Exception as exc:
        try: db.connection.rollback()
        except Exception: pass
        queryable=False; error=type(exc).__name__
    probes.append({'table':qualified,'queryable':queryable,'permno_col':permno_col,'observation_col':observation_col,'start_col':start_col,'end_col':end_col,'error':error})
probe_audit=pd.DataFrame(probes);probe_audit.to_parquet(MANIFESTS/'sp500_source_probe.parquet',index=False);display(probe_audit)
if not valid: raise RuntimeError('Candidate tables exist but none has a recognized point-in-time membership structure.')
source=sorted(valid,key=lambda x:(x['table']!='crsp_a_indexes.dsp500list',x['table']))[0]
print('Selected point-in-time source:',source)

## 5. Extract quarter-specific membership

For dated constituent files, the latest available index date on or before each calendar quarter-end is used. This correctly handles weekends and holidays.

In [ ]:
rows=[]; dates=[]
for v in vintages.itertuples(index=False):
    if source['observation_col']:
        d=db.raw_sql(f"select max({source['observation_col']}) as index_date from {source['table']} where {source['observation_col']} <= '{v.cutoff.date()}'",date_cols=['index_date']).index_date.iloc[0]
        frame=db.raw_sql(f"select {source['permno_col']} as permno from {source['table']} where {source['observation_col']} = '{pd.Timestamp(d).date()}'")
    else:
        d=v.cutoff
        frame=db.raw_sql(f"select {source['permno_col']} as permno from {source['table']} where {source['start_col']} <= '{v.cutoff.date()}' and ({source['end_col']} is null or {source['end_col']} >= '{v.cutoff.date()}')")
    frame['permno']=pd.to_numeric(frame.permno,errors='coerce').astype('Int64');frame=frame.dropna().drop_duplicates();frame['vintage']=v.label;frame['cutoff']=v.cutoff;frame['index_date']=pd.Timestamp(d)
    rows.append(frame);dates.append({'vintage':v.label,'cutoff':v.cutoff,'index_date':pd.Timestamp(d),'securities':frame.permno.nunique()})
membership_security=pd.concat(rows,ignore_index=True);membership_security.to_parquet(PROCESSED/'sp500_security_membership_2024q1_q4.parquet',index=False)
display(pd.DataFrame(dates))

## 6. Map securities to firms using the existing point-in-time CRSP snapshots

In [ ]:
mapped=[]
for v in vintages.itertuples(index=False):
    year=int(v.label[:4]);quarter=int(v.label[-1])
    p=ROOT/'data'/'crosswalks'/'crsp'/f'year={year}'/f'quarter={quarter}'/f'stocknames_{v.label}.parquet'
    names=pd.read_parquet(p); names['permno']=pd.to_numeric(names.permno,errors='coerce').astype('Int64')
    names=names.sort_values(['permno','nameenddt']).drop_duplicates('permno',keep='last')
    q=membership_security[membership_security.vintage==v.label].merge(names,on='permno',how='left',validate='one_to_one')
    q['vintage']=v.label;mapped.append(q)
membership=pd.concat(mapped,ignore_index=True)
membership['permco']=pd.to_numeric(membership.permco,errors='coerce').astype('Int64')
mapping_audit=(membership.groupby('vintage',as_index=False).agg(index_securities=('permno','nunique'),mapped_securities=('permco','count'),firms=('permco','nunique'),duplicate_firm_securities=('permco',lambda x:int(x.notna().sum()-x.nunique()))))
mapping_audit['permco_mapping_rate']=mapping_audit.mapped_securities/mapping_audit.index_securities
display(mapping_audit)
membership.to_parquet(PROCESSED/'sp500_point_in_time_membership_with_crsp_ids.parquet',index=False);mapping_audit.to_parquet(RESULTS/'membership_mapping_audit.parquet',index=False)

## 7. Audit quarterly embedding coverage

Missing embedding firms remain documented and are not replaced after exposure labels are observed.

In [9]:
embedding_rows=[]; covered=[]
for v in vintages.itertuples(index=False):
  firms=membership[membership.vintage==v.label].dropna(subset=['permco']).drop_duplicates('permco')
  for model in design['representations']:
    p=ROOT/'active_passive_extension'/'benchmarks'/'all_s12_control'/v.label/model/'asset_embeddings_10d.parquet'
    e=pd.read_parquet(p); e['permco']=pd.to_numeric(e.permco,errors='coerce').astype('Int64')
    found=firms.merge(e[['permco','firm_name']],on='permco',how='left',indicator=True)
    embedding_rows.append({'vintage':v.label,'model':model,'sp500_firms':len(firms),'embedded_firms':int((found._merge=='both').sum()),'coverage':float((found._merge=='both').mean()),'sha256':sha256(p)})
    if model=='rs_l_min': covered.append(found[found._merge=='both'].drop(columns='_merge'))
embedding_audit=pd.DataFrame(embedding_rows);embedding_audit.to_parquet(RESULTS/'embedding_coverage_audit.parquet',index=False);display(embedding_audit)

,vintage,model,sp500_firms,embedded_firms,coverage,sha256
0,2024q1,rs_binary,500,411,0.822000,f05f53385ecf6966eb73b2b9338855a80d556a8498d83a...
1,2024q1,rs_l_min,500,411,0.822000,7ecbac52e30387341848faab5ca6f69cbf6faf856f008a...
2,2024q2,rs_binary,500,407,0.814000,369694d72f0e440178708fcc91a8aec250f7bbafeb74c6...
3,2024q2,rs_l_min,500,407,0.814000,b33176eb9373a98cc6b0254d8b4143258a92a268214dfe...
4,2024q3,rs_binary,501,403,0.804391,fa244d5419bc1398190f4f9dfcae24cf190d47927baec8...
5,2024q3,rs_l_min,501,403,0.804391,d62f8eca4d408f59fbd3a55b10204c88ca73d73a6669d1...
6,2024q4,rs_binary,500,410,0.820000,d9ddbf00039b4bfbb4daec34c4baaf4ba942733d97fa04...
7,2024q4,rs_l_min,500,410,0.820000,72cb833910f71462b45b1bfbb2192ae5179b6abe160400...


## 8. Construct a robust CIK bridge

The primary route is the CRSP/Compustat link history followed by Compustat company CIK. Exact normalized SEC ticker matching is retained only as a documented fallback.

In [15]:
sp500_permnos=sorted(membership.permno.dropna().astype(int).unique())
permno_sql=','.join(map(str,sp500_permnos))
ccm_candidates=['crsp_a_ccm.ccmxpf_lnkhist','crsp.ccmxpf_lnkhist'];link=None
for table in ccm_candidates:
    try:
        candidate=db.raw_sql(f"select gvkey,lpermno,linkdt,linkenddt,linktype,linkprim from {table} where lpermno in ({permno_sql})",date_cols=['linkdt','linkenddt'])
        candidate['linktype']=candidate.linktype.astype('string').str.strip().str.upper()
        candidate['linkprim']=candidate.linkprim.astype('string').str.strip().str.upper()
        candidate=candidate[candidate.linktype.isin(['LC','LU']) & candidate.linkprim.isin(['P','C'])].copy()
        if candidate.empty: raise ValueError('No eligible LC/LU primary CCM links returned')
        link=candidate; print('CCM source:',table,'eligible links:',len(link)); break
    except Exception:
        try: db.connection.rollback()
        except Exception: pass
if link is None: raise RuntimeError('CCM link unavailable for the S&P 500 PERMNOs; do not proceed to SEC acquisition.')
try: company=db.raw_sql('select gvkey,cik as compustat_cik,conm from comp.company')
except Exception:
    try: db.connection.rollback()
    except Exception: pass
    raise RuntimeError('Compustat company identifiers are not queryable; do not proceed to SEC acquisition.')
link['gvkey']=link.gvkey.astype('string').str.strip().str.zfill(6)
company['gvkey']=company.gvkey.astype('string').str.strip().str.zfill(6)
link['lpermno']=pd.to_numeric(link.lpermno,errors='coerce').astype('Int64')
bridge_rows=[]
if link is not None and len(company):
    link['linkenddt']=pd.to_datetime(link.linkenddt).fillna(pd.Timestamp('2099-12-31'))
    for v in vintages.itertuples(index=False):
        q=membership[membership.vintage==v.label].copy(); active=link[(link.linkdt<=v.cutoff)&(link.linkenddt>=v.cutoff)].copy()
        active['link_priority']=active.linkprim.map({'P':0,'C':1}).fillna(2)
        z=q.merge(active,left_on='permno',right_on='lpermno',how='left').merge(company,on='gvkey',how='left')
        z=z.sort_values(['permno','link_priority','linktype'],ascending=[True,True,True]).drop_duplicates('permno');z['vintage']=v.label;bridge_rows.append(z)
bridge=pd.concat(bridge_rows,ignore_index=True) if bridge_rows else membership.copy()
bridge['cik']=pd.to_numeric(bridge['compustat_cik'],errors='coerce').astype(pd.Int64Dtype()) if 'compustat_cik' in bridge else pd.Series(pd.NA,index=bridge.index,dtype='Int64')
bridge.to_parquet(PROCESSED/'sp500_point_in_time_identifier_bridge.parquet',index=False)
firm_bridge=bridge.sort_values(['vintage','permco','cik'],na_position='last').drop_duplicates(['vintage','permco'])
cik_audit=(firm_bridge.groupby('vintage',as_index=False).agg(sp500_firms=('permco','nunique'),firms_with_cik=('cik','count'),unique_ciks=('cik','nunique')));cik_audit['cik_mapping_rate']=cik_audit.firms_with_cik/cik_audit.sp500_firms
display(cik_audit);cik_audit.to_parquet(RESULTS/'cik_coverage_audit.parquet',index=False)

CCM source: crsp_a_ccm.ccmxpf_lnkhist eligible links: 612


,vintage,sp500_firms,firms_with_cik,unique_ciks,cik_mapping_rate
0,2024q1,500,497,497,0.994
1,2024q2,500,497,497,0.994
2,2024q3,501,498,498,0.994012
3,2024q4,500,497,497,0.994


## 9. Forecast document volume, runtime, and API spending

This is a planning bound, not an authorization to spend. Deduplication by accession and content hash, retrieval before extraction, and a 25-document guarded batch remain mandatory.

In [16]:
firm_vintages=bridge.dropna(subset=['permco','cik'])[['vintage','permco','cik']].drop_duplicates()
unique_firms=firm_vintages.permco.nunique();upper_unique_documents=unique_firms*4
scenarios=[]
for retrieved_share in [0.25,0.50,1.00]:
  for tokens_per_document in [1500,4500]:
    documents=int(np.ceil(upper_unique_documents*retrieved_share));input_millions=documents*tokens_per_document/1e6;output_millions=documents*800/1e6
    scenarios.append({'retrieval_positive_share':retrieved_share,'documents_sent_to_api':documents,'input_tokens_per_document':tokens_per_document,'estimated_input_million_tokens':input_millions,'estimated_output_million_tokens':output_millions,'estimated_minutes_at_6_concurrent_and_20_seconds_each':documents*20/6/60})
cost_scenarios=pd.DataFrame(scenarios);cost_scenarios.to_parquet(RESULTS/'pre_metadata_api_volume_scenarios.parquet',index=False)
display(pd.DataFrame([{'unique_sp500_firms_with_cik':unique_firms,'maximum_unique_10k_10q_documents':upper_unique_documents,'guarded_api_pilot_documents':design['decision_gates']['api_pilot_documents'],'full_api_run_enabled':False}]))
display(cost_scenarios)
print('Dollar cost will be calculated after the model and its then-current prices are frozen; no stale price is hard-coded here.')

,unique_sp500_firms_with_cik,maximum_unique_10k_10q_documents,guarded_api_pilot_documents,full_api_run_enabled
0,512,2048,25,False


,retrieval_positive_share,documents_sent_to_api,input_tokens_per_document,estimated_input_million_tokens,estimated_output_million_tokens,estimated_minutes_at_6_concurrent_and_20_seconds_each
0,0.25,512,1500,0.768,0.4096,28.444444
1,0.25,512,4500,2.304,0.4096,28.444444
2,0.50,1024,1500,1.536,0.8192,56.888889
3,0.50,1024,4500,4.608,0.8192,56.888889
4,1.00,2048,1500,3.072,1.6384,113.777778
5,1.00,2048,4500,9.216,1.6384,113.777778


Dollar cost will be calculated after the model and its then-current prices are frozen; no stale price is hard-coded here.


## 10. Decision gates and manifest

Failure of a gate is diagnostic. It is not permission to hand-select replacement firms.

In [17]:
g=design['decision_gates']
gates=pd.DataFrame([
 {'gate':'minimum_constituent_firms','value':int(mapping_audit.firms.min()),'threshold':g['minimum_point_in_time_sp500_firms_per_vintage'],'passed':bool(mapping_audit.firms.min()>=g['minimum_point_in_time_sp500_firms_per_vintage'])},
 {'gate':'permco_mapping_rate','value':float(mapping_audit.permco_mapping_rate.min()),'threshold':g['minimum_permco_mapping_rate'],'passed':bool(mapping_audit.permco_mapping_rate.min()>=g['minimum_permco_mapping_rate'])},
 {'gate':'cik_mapping_rate','value':float(cik_audit.cik_mapping_rate.min()),'threshold':g['minimum_cik_mapping_rate'],'passed':bool(cik_audit.cik_mapping_rate.min()>=g['minimum_cik_mapping_rate'])},
 {'gate':'outcomes_unopened','value':0.0,'threshold':0.0,'passed':True},
 {'gate':'api_calls_not_made','value':0.0,'threshold':0.0,'passed':True}])
gates.to_parquet(MANIFESTS/'universe_decision_gates.parquet',index=False);display(gates)
artifact_paths=[PROCESSED/'sp500_security_membership_2024q1_q4.parquet',PROCESSED/'sp500_point_in_time_membership_with_crsp_ids.parquet',PROCESSED/'sp500_point_in_time_identifier_bridge.parquet']
manifest={'stage':'sp500_point_in_time_universe','design_sha256':design_hash,'sp500_source':source,'artifacts':[{'path':str(p.relative_to(EXT)),'sha256':sha256(p)} for p in artifact_paths],'all_gates_passed':bool(gates.passed.all()),'outcomes_accessed':False,'filings_downloaded':False,'api_calls_made':False}
manifest['sha256']=hashlib.sha256(json.dumps(manifest,sort_keys=True,default=str).encode()).hexdigest()
(MANIFESTS/'universe_manifest.json').write_text(json.dumps(manifest,indent=2,sort_keys=True,default=str)+'\n')
print('All gates passed:',manifest['all_gates_passed']);print('Manifest:',manifest['sha256'])
if not gates.passed.all(): print('STOP: inspect failed coverage gates before building the SEC corpus.')

,gate,value,threshold,passed
0,minimum_constituent_firms,500.000,490.00,True
1,permco_mapping_rate,1.000,0.99,True
2,cik_mapping_rate,0.994,0.95,True
3,outcomes_unopened,0.000,0.00,True
4,api_calls_not_made,0.000,0.00,True


All gates passed: True
Manifest: c046d21edafdeab0c9c08995343b455749e0bca8ab52fd666c865d2f3546666c
